# Studio di MERGE
basato su cleaning 4
## Inizializzazione ed Import

In [ ]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from data_model.manage_excel_support_file import *
from data_model.MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake


**Mixed info**\
'ADNIMERGE', \
'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

**Single Cofactor**\
'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

**Volumes**\
'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'
'UCSDVOL', 'UPENN_ROI_MARS',  --> do NOT use FreeSurfer but other model or Atlas so not comparable

**CSF**\
'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [ ]:
file_codes =['ADNIMERGE', 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA'] #
search = client.query_files(
    query={'custom.level' : 'cleaned_04', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(len(zip_files))

In [ ]:
##### MODIFICARE 
# choose from: volumes, scale, csf, plasma, pet, cofactor
category = 'scale'

In [ ]:
dfs = {}
df_names = {}
df_code = []
idx = 0
for file_name, df_raw in zip_files.items():
    print('\n### ', file_name)
    df_copy = df_raw.copy(deep=True)
    len_pre = len(df_copy.columns)
    row_pre = len(df_copy)
    # get the sub_df focusing on the category chosen
    df_copy = mergeTools.filter_df_category(df_copy, category)
    df_copy = df_copy.drop_duplicates()
    len_post = len(df_copy.columns)
    row_post = len(df_copy)
    if df_copy.empty:
        print(file_name, f'------> non ha colonne dellca categoria {category}')
        continue
    # get the common columns among all the dfs
    if idx == 0:
        dfs_columns = set(df_copy.columns)
    else: 
        dfs_columns &= set(df_copy.columns)
    # Ensure EXAMDATE in date format and correct order of the dfs
    df_copy['EXAMDATE'] = pd.to_datetime(df_copy['EXAMDATE'])
    df_copy = df_copy.sort_values(by=['RID', 'EXAMDATE']).reset_index(drop=True)
    if 'FSVERSION' in df_copy.columns:
        df_copy['FSVERSION'] = df_copy['FSVERSION'].astype(str)  
    # aggiornamento liste e dizionari
    dfs[f"df_{idx}"] = df_copy  
    df_names[f"df_{idx}"] = file_name 
    df_code.append(f"df_{idx}")
    # definizione variabile df
    globals()[f"df_{idx}"] = df_copy
    print(idx, '--->', file_name, '\t\t\t### ', len_post, '/', len_pre, '\n\t\t\t\t\t\t rows: ', row_post, '/', row_pre)
    idx += 1

print(f'===============================================================================\n\nColumns common to all df:\n\n{dfs_columns}\n\n===============================================================================')

time_buffer = pd.Timedelta(days=80)

subj_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID'])
print("righe con stessi ####### RID:")
display(subj_matrix)
        
subj_date_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE'], time_buffer=time_buffer)
print("righe con stessi ####### RID-EXAMDATE: --> time_buffer=", time_buffer)
display(subj_date_matrix)

if set(['FSVERSION', 'IMAGEUID']).issubset(set(dfs_columns)):
    subj_viscode_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE', 'FSVERSION','IMAGEUID'])
    print("righe con stessi ####### RID-EXAMDATE-FSVERSION-IMAGEUID: --> time_buffer=", time_buffer)
    display(subj_viscode_matrix)

if set('METHOD').issubset(set(dfs_columns)):
    subj_viscode_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE', 'METHOD'])
    print("righe con stessi ####### RID-EXAMDATE-METHOD: --> time_buffer=", time_buffer)
    display(subj_viscode_matrix)

In [ ]:
# SEVUOI CONTROLLARE QUALCOSA, tipo se cè una variabile o quali sono i valori di quella colonna
x = 0
for df_x in dfs.values():
    print(x, 'update_stamp' in df_x.columns)
    x += 1

# Inizio Merge
## Definizione di df_base e Gerarchia di DF da mergiare
Come primo df_base scegliere o ADNIMERGE o l'altro df con più match.\
Quindi per la gerarchia un opzione è scegliere prima gli altri df con righe in comune e quindi quelli senza righe in comune.\
Se il base non è ADNIMERGE, metterlo come ultimo MA attenzione nel merge in quel caso inveritre e mettere (df_add, df_base, ...)

In [ ]:
############   MODIFICARE
base = 'df_0'
df_base = dfs[base].copy(deep=True)
idx_add = ['df_6', 'df_1', 'df_2', 'df_3', 'df_4', 'df_5']
merge_contains = [base]
sub_with_match = set()
time_buffer = pd.Timedelta(days=80)
i = 0
print(f'df_base: {base} ->    ', df_names[base].split('_')[0])

## Difine DF_ADD and connection with DF_BASE

In [ ]:
df_add = dfs[idx_add[i]].copy(deep=True)
df_add_name = df_names[idx_add[i]]
print(idx_add[i], '->\t',df_add_name.split('_')[0])

##########################################################################################################

exact_matches, buffer_matches = mergeTools.find_visit_matches(df_base, df_add, buffer_days=time_buffer)
mergeTools.verify_visit_matches(exact_matches, buffer_matches)
exact_index1, exact_index2 = mergeTools.list_index_visit_matches(exact_matches)
buff_index1, buff_index2 = mergeTools.list_index_visit_matches(buffer_matches)
all_index1 = exact_index1.union(buff_index1)
all_index2 = exact_index2.union(buff_index2)

columns_in_common = list(df_base.columns.intersection(df_add.columns))
columns_only_base = list(df_base.columns.difference(df_add.columns))
columns_only_add = list(df_add.columns.difference(df_base.columns))


print('\n\nExact matches: \t\t', len(exact_index1))
print('Buffered matches: \t', len(buff_index1))
print('Overall matches: \t', len(all_index1))

if len(buff_index1) != len(buff_index2):
    print('\n ------>>> ATTENZIONE: indici match con buffer SPAIATI')

if len(exact_index1) != len(exact_index2):
    print('\n ------>>> ATTENZIONE: indici match esatti SPAIATI')

if len(all_index1) != len(all_index2):
    print('\n ------>>> ATTENZIONE: indici match globali SPAIATI')

print('\n===============================================================================\n')

if not columns_only_add and not columns_only_base:
    print(f'The 2 df have ALL columns in common:\n{columns_in_common}\n\n')
else:
    print(f'The 2 df hava some colums NOT in common:\n\nOnly in BASE: {columns_only_base}\nOnly in ADD: {columns_only_add}\n\n')

diff_date = False
if all_index1.empty and all_index2.empty:
        print('NO matches ==> SMOOTH MERGE')
elif not all_index1.empty and not all_index2.empty:
    if buff_index1.empty and buff_index2.empty:
        print('Just EXACT matches ==> move to merge')
    else: 
        diff_date = True
        print('There are BUFFERED matches ==> first check buffered matches')

In [ ]:
modify_examdate = False
if diff_date:
    print('df_add --> ', df_names[idx_add[i]], '\ndf_base --> ', [df_names[x] for x in merge_contains])
    col_list = list(df_base.columns) + [c for c in df_add.columns if c not in df_base.columns]
    
    temp_merge = mergeTools.create_temp_merge(df_base, df_add, buff_index1, buff_index2, col_list=col_list)
    diff = temp_merge['EXAMDATE_1']-temp_merge['EXAMDATE_2']
    display(diff[diff != pd.Timedelta(days=0)])
    print(len(diff[diff != pd.Timedelta(days=0)]))
    display(temp_merge.loc[diff[diff != pd.Timedelta(days=0)].index])
    modify_examdate = True
else:
    print('No bufferd matches')
    

In [ ]:
# best practice automatica
index_modified = []
idx_manual_check = []
if modify_examdate:
    is_adnimerge = 'ADNIMERGE' in df_add_name
    print('We are adding ADNIMERGE') if is_adnimerge else print('We are adding a regular df')
    for x1, x2 in zip(buff_index1, buff_index2):
        if df_base.loc[x1, 'VISCODE'] != df_add.loc[x2, 'VISCODE'] and ('m' in df_base.loc[x1, 'VISCODE'] and 'm' in df_add.loc[x2, 'VISCODE']):
            idx_manual_check.append([x1,x2])
        
        elif df_base.loc[x1, 'VISCODE'] == df_add.loc[x2, 'VISCODE'] or abs(df_base.loc[x1, 'EXAMDATE'] - df_add.loc[x2, 'EXAMDATE']) < pd.Timedelta(days=30):
            if is_adnimerge:
                df_base.loc[x1, 'EXAMDATE'] = df_add.loc[x2, 'EXAMDATE']
            else:
                df_add.loc[x2, 'EXAMDATE'] = df_base.loc[x1, 'EXAMDATE']
            index_modified.append([x1, x2])

    print('Rows modified', len(index_modified))
    print('Need to check', len(idx_manual_check), 'matches')
else:
    print('No modification of EXAMDATE')

In [ ]:
# in case o NEED TO CHECK MATCHES
'''
idx_manual_check

# nuova cella 
idx = idx_manual_check[0]
rid = df_base.loc[idx[0], 'RID']
pd.DataFrame([df_base.loc[idx[0]], df_add.loc[idx[1]]])

# nuova cella 
df_base[df_base['RID']==rid]

# nuova cella 
df_add[df_add['RID']==rid]

# nuova cella 
df_add.loc[3311, 'EXAMDATE'] = df_base.loc[7102, 'EXAMDATE']
'''

## MERGING PHASE

In [ ]:
if 'ADNIMERGE' in df_add_name:
    print('ADNIMERGE added to the \n')
    df_merged = mergeTools.get_merged_df(df_add, df_base, category=category)
else:
    df_merged = mergeTools.get_merged_df(df_base, df_add, category=category)


In [ ]:

df_base = df_merged.copy(deep=True)
merge_contains.append(idx_add[i])
#prepare for next merge
i += 1
print(f'\nil merge contine i seguenti df: {merge_contains}')
if i <= len(idx_add)-1:
    print(f'il prossimo df da unire è: {idx_add[i]}\n\n ===> torna al capitolo: "Difine DF_ADD and connection with DF_BASE"')
else:
    print('FINISHED MERGE!!!!!')
    if 'METHOD' in df_merged.columns:
        new_col_name = 'METHOD_' + category.upper()
        df_merged = df_merged.rename(columns={'METHOD': new_col_name})
        print('METHOD column renamed to '+ new_col_name)

# SALVAREEEE

In [ ]:
df_merged = df_merged.sort_values(by=['RID', 'EXAMDATE']).reset_index(drop=True)
df_merged = df_merged.calculate_visit_month(df_merged)

In [ ]:
#check prima di salvare nel DL
df_merged.head(50)

In [ ]:
file_codes = {'volumes': 'VOLMERGE', 'scale': 'SCALEMERGE', 'csf': 'CSFMERGE', 'plasma': 'PLMERGE', 'pet': 'PETMERGE', 'cofactor': 'COFMERGE'}

new_file_name = category.upper() + '_merged.csv'
print('New merge file name: ', new_file_name)
file_code = file_codes[category]

# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=df_merged,
    object_name=new_file_name,
    prefix='cleaned/merged/category',
    metadata={
        'level': 'merged',
        'file_code': file_code,
        'source': 'ADNI'
    }
)